In [1]:
# Library imports
from dotenv import load_dotenv
import os
import openai
import pandas as pd
import json
import time
from typing import Dict, List

In [2]:
# Load OpenAI API key from .env file
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

In [3]:
# Policy characteristic definitions
mission_oriented_policy = {
    "budget_structure": "Consists of large-scale contracts with significant funding allocations.",
    "schedule": "Long-term schedules are assigned with no allowance for mid-course failure.",
    "development_cost": {
        "funding": "Fully funded by the government.",
        "payment_structure": "Government pays primary development costs.",
        "contract_type": "Cost-plus contracts, with the government covering any cost overruns."
    },
    "mass_production": "Production is government-owned, operated, maintained, and utilized.",
    "contract_form": {
        "type": "Traditional government contracts under the Federal Acquisition Regulation (FAR).",
        "development_contracts": "Major development and production contracts are issued as cost-plus contracts."
    },
    "role_distribution": {
        "lead": "Government-led development with total management responsibility.",
        "industry_role": "Private companies act as subcontractors manufacturing parts and systems.",
        "specs": "Government defines all technical specifications and design requirements."
    },
    "ownership_and_operations": {
        "assets": "Developed rockets and spacecraft are owned by the government.",
        "mission_operations": "Conducted under government supervision with limited commercial use."
    },
    "program_management": {
        "approach": "Strict supervision and process-centered management.",
        "collaboration": "Close collaboration between the government and contractors.",
        "review_approval": "Government review and approval required at every stage."
    },
    "design_approach": "Designs are tailored exactly to requirements, reflecting government and agency standards.",
    "development_risk": "Solely borne by the government, including cost overruns and schedule delays.",
    "operational_risk": "Government is responsible for losses and crew casualties in the event of mission failure.",
    "commercial_risk": {
        "market_relation": "Independent of market conditions.",
        "compensation": "Government provides compensation.",
        "market_use": "Launch vehicles are government-exclusive systems, unrelated to the commercial market.",
        "profit": "Contractors receive predetermined profit from the government."
    },
    "intellectual_property": {
        "accumulation": "Technology and know-how accumulated within government and its contractors.",
        "ownership": "Patents and design information are controlled by the government.",
        "transfer": "Only selected technologies are transferred to the private sector."
    },
    "technology_transfer": {
        "method": "Advanced technologies developed by the government are disseminated through spinoff programs.",
        "private_sector_role": "Private entities apply technologies post-development rather than directly participating in R&D."
    }
}

diffusion_oriented_policy = {
    "budget_structure": "Composed of multiple small-scale contracts with limited funding per unit.",
    "schedule": "Short-term schedules are set, and failure to meet key milestones results in contract termination.",
    "development_cost": {
        "funding": "Cost-sharing between government and private sector.",
        "payment_structure": "Government provides fixed funding; contractors must invest their own capital.",
        "contract_type": "Performance-based contracts, payment issued only upon milestone achievement."
    },
    "mass_production": "Ownership, operation, and maintenance lie with private companies. Government purchases services as needed.",
    "contract_form": {
        "type": "Agreements under space law, allowing flexibility.",
        "development_contracts": "Issued as fixed-price contracts to private entities."
    },
    "role_distribution": {
        "lead": "Joint development between government and private companies.",
        "government_role": "Sets high-level goals and funding; provides technical advice and evaluation only.",
        "industry_role": "Private firms lead design and development efforts."
    },
    "ownership_and_operations": {
        "assets": "Rockets and spacecraft are owned by private companies.",
        "mission_operations": "Government uses them as a service; private firms may serve commercial clients as well."
    },
    "program_management": {
        "approach": "Light supervision with performance-based oversight.",
        "government_involvement": "Validates major milestone achievements only.",
        "decision_making": "Detailed design and development decisions are at company discretion."
    },
    "design_approach": "Designs are goal-oriented and flexible, with multiple approaches allowed. Government specifies outcomes, not methods.",
    "development_risk": {
        "burden": "Shared between government and private companies.",
        "failure_consequence": "Government withholds remaining support; remaining losses are borne by the company."
    },
    "operational_risk": {
        "responsibility": "Private sector under commercial launch oversight agencies.",
        "failure_consequence": "Contractors bear full responsibility for re-launch costs in case of mission failure."
    },
    "commercial_risk": {
        "burden": "Fully borne by the private sector.",
        "ownership": "Launch vehicles are privately owned.",
        "market_responsibility": "Private companies are responsible for securing commercial demand.",
        "revenue_opportunity": "Firms may earn additional revenue from launching third-party satellites."
    },
    "intellectual_property": {
        "ownership": "Technology and know-how are accumulated within private firms.",
        "government_access": "Government retains limited access to necessary data only.",
        "control": "Intellectual property rights remain with the private companies."
    },
    "technology_transfer": {
        "development": "Private firms develop technologies to meet government needs.",
        "government_support": "Government may provide test facilities and technical expertise when necessary.",
        "utilization": "Firms retain and commercialize the technologies after development."
    }
}

In [4]:
def load_data():
    """Data loading function"""
    try:
        # Government agency data (ID 1-4)
        gov_data = pd.read_csv("trait_data_gov.csv")
        
        # Company data (ID 5-106)
        com_data = pd.read_csv("trait_data_com.csv")
        
        # University data (ID 107-146)
        univ_data = pd.read_csv("trait_data_univ.csv")
        
        print(f"Government data: {len(gov_data)} entities loaded")
        print(f"Company data: {len(com_data)} entities loaded")
        print(f"University data: {len(univ_data)} entities loaded")
        
        return gov_data, com_data, univ_data
    
    except FileNotFoundError as e:
        print(f"File not found: {e}")
        return None, None, None


In [5]:
def ask_openai(system_message, user_message, max_retries=3):
    """GPT response generation function"""
    for attempt in range(max_retries):
        try:
            response = openai.ChatCompletion.create(
                model="gpt-4.1",
                messages=[
                    {"role": "system", "content": system_message},
                    {"role": "user", "content": user_message},
                ],
                temperature=1.0,
                max_tokens=4000
            )
            return response.choices[0].message.content.strip()
        
        except Exception as e:
            print(f"API call failed (attempt {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2)  # Wait before retry
            else:
                return "API_ERROR"

In [6]:
def create_system_message(row, data_type):
    """Create system message for each ID"""
    if data_type == "gov":
        return f"""You are a decision-maker for government agency ID {row['ID']}, {row['Name']}. Your government agency has the following characteristics: Category is {row['Category']}, established in {row['Year']}, has {row['Employees']} employees, yearly budget is {row['Budget']}, organization information is {row['Organization Info']}, main mission is {row['Mission']}, subordinate centers are {row['Subordinate Centers']}, total number of papers published is {row['Papers']}, number of space technology-related patents (B64G) is {row['Patents']}, and headquarters location is {row['Location']}."""
    
    elif data_type == "com":
        return f"""You are a decision-maker for company ID {row['ID']}, {row['Name']}. Your company has the following characteristics: Main industry is {row['Main Industry']}, sub industry is {row['Sub Industry']}, established in {row['Year']}, has {row['Employees']} employees, company type is {row['Company Type']}, revenue is {row['Revenue']}, operating income is {row['Operating Income']}, listing status is {row['Listed']}, enterprise value is {row['Enterprise Value']}, funding level is {row['Funding']}, company information is {row['Company Info']}, main products and services are {row['Products Services']}, core technology is {row['Core Technology']}, number of space technology-related patents (B64G) is {row['Patents']}, and headquarters location is {row['Location']}."""
    
    elif data_type == "univ":
        return f"""You are a decision-maker for university ID {row['ID']}, {row['Name']}. Your university has the following characteristics: Category (private/public) is {row['Category']}, established in {row['Year']}, annual R&D expenditure is {row['R&D Expenditure']}, total number of aerospace department professors is {row['Professors']}, number of aerospace graduate students is {row['Graduate Students']}, representative laboratory is {row['Laboratory']}, main research area is {row['Research Area']}, core strength of the aerospace department is {row['Core Strength']}, total number of papers published is {row['Papers']}, number of space technology-related patents (B64G) is {row['Patents']}, and university location is {row['Location']}."""

In [7]:
def create_partner_info(gov_data, com_data, univ_data):
    """Generate text information for all potential collaboration partners"""
    partner_info = "The following is information about potential collaboration partners:\n\n"
    
    # Government agency information
    partner_info += "=== Government Agencies ===\n"
    for _, row in gov_data.iterrows():
        partner_info += f"ID {row['ID']}: {row['Name']} - Category: {row['Category']}, Established: {row['Year']}, Employees: {row['Employees']}, Yearly budget: {row['Budget']}, Organization Info: {row['Organization Info']}, Mission: {row['Mission']}, Subordinate Centers: {row['Subordinate Centers']}, Papers: {row['Papers']}, Patents: {row['Patents']}, Location: {row['Location']}\n"
    
    # Company information
    partner_info += "\n=== Companies ===\n"
    for _, row in com_data.iterrows():
        partner_info += f"ID {row['ID']}: {row['Name']} - Industry: {row['Main Industry']}/{row['Sub Industry']}, Established: {row['Year']}, Employees: {row['Employees']}, Type: {row['Company Type']}, Revenue: {row['Revenue']}, Operating Income: {row['Operating Income']}, Listed: {row['Listed']}, Enterprise Value: {row['Enterprise Value']}, Funding: {row['Funding']}, Company Info: {row['Company Info']}, Main Product & Service: {row['Products Services']}, Core Technology: {row['Core Technology']}, Patents: {row['Patents']}, Location: {row['Location']}\n"
    
    # University information
    partner_info += "\n=== Universities ===\n"
    for _, row in univ_data.iterrows():
        partner_info += f"ID {row['ID']}: {row['Name']} - Category(Public/Private): {row['Category']}, Established: {row['Year']}, R&D Expenditure: {row['R&D Expenditure']}, Professors: {row['Professors']}, Graduate Students: {row['Graduate Students']}, Main Laboratory: {row['Laboratory']}, Research Area: {row['Research Area']}, Core Stength: {row['Core Strength']}, Papers: {row['Papers']}, Patents: {row['Patents']}, Location: {row['Location']}\n"
    
    return partner_info

In [8]:
def create_user_prompt(policy_type, policy_dict, partner_info):
    """Generate user prompt"""
    policy_str = json.dumps(policy_dict, indent=2, ensure_ascii=False)
    
    base_prompt = f"""
The government will be operating research and development and will be forming markets in the following areas: 
Launch Services, Launch Site Operators, Launch Support Services and Equipment, Launch Vehicle Manufacturers, 
Launch Vehicle Equipment and Software, Manufacturers of Spacecraft Deployment Systems, 
Launch Vehicle Test Equipment, Launch Vehicle Design and Manufacturing Support. 

The policy direction includes: {policy_str}  
The profiles of potential collaboration partners: {partner_info}  

Based on the policy characteristics and the profiles of potential collaboration partners, 
which partners (by ID) would you choose to collaborate with (you can select 0 to 146 partners)?  

Please respond using the following format:  
- Selected Partner ID(s): (respond with numbers only, comma-separated if more than one)  
- Reason(s) for Selecting Each Partner ID:  
- Overall Considerations for Your Decision:
"""
    
    return base_prompt

In [9]:
def process_all_entities():
    """Perform analysis for all entities"""
    # Load data
    gov_data, com_data, univ_data = load_data()
    
    if gov_data is None or com_data is None or univ_data is None:
        print("Failed to load data.")
        return
    
    # Generate collaboration partner information
    partner_info = create_partner_info(gov_data, com_data, univ_data)
    
    # List to store results
    results = []
    
    # Combine all data into one list
    all_data = []
    
    # Process government agencies
    for _, row in gov_data.iterrows():
        all_data.append((row, "gov"))
    
    # Process companies
    for _, row in com_data.iterrows():
        all_data.append((row, "com"))
    
    # Process universities
    for _, row in univ_data.iterrows():
        all_data.append((row, "univ"))
    
    # Perform analysis for each entity
    for i, (row, data_type) in enumerate(all_data):
        print(f"Processing: ID {row['ID']} ({i+1}/{len(all_data)})")
        
        # Generate system message
        system_message = create_system_message(row, data_type)
        
        # Question for mission-oriented policy
        mission_prompt = create_user_prompt("mission-oriented", mission_oriented_policy, partner_info)
        mission_response = ask_openai(system_message, mission_prompt)
        
        # Question for diffusion-oriented policy
        diffusion_prompt = create_user_prompt("diffusion-oriented", diffusion_oriented_policy, partner_info)
        diffusion_response = ask_openai(system_message, diffusion_prompt)
        
        # Store results
        results.append({
            "ID": row['ID'],
            "M_Partner": mission_response,
            "D_Partner": diffusion_response
        })
        
        # Control API call interval
        time.sleep(1)
    
    # Convert results to DataFrame and save
    answer_sheet = pd.DataFrame(results)
    answer_sheet.to_csv("answer_sheet.csv", index=False, encoding='utf-8-sig')
    print("Answer sheet saved to 'answer_sheet.csv'")
    
    return answer_sheet